# Agente MIP — Programación Entera Mixta

In [54]:
import os
import pulp
import json
import sys
from dotenv import load_dotenv

# Cargar variables de entorno desde un archivo .env (si existe)
load_dotenv()

# Obtener API key desde las variables de entorno
API_KEY = os.getenv("API_KEY")

def crear_cliente_gemini(api_key: str):
    try:
        from google import genai
    except Exception:
        return None
    try:
        client = genai.Client(api_key=api_key)
        return client
    except Exception:
        return None
def consultar_agente_ia(client, resultados: dict, nombre_problema: str) -> str:
    """Envía los resultados al agente IA y solicita un análisis profundo y accionable.

    El agente debe:
    - Verificar numéricamente que la solución respeta todas las restricciones y reportar cualquier violación.
    - Identificar las restricciones "binding" (activas) y explicar por qué son determinantes.
    - Proveer una explicación intuitiva de la solución (cómo se asignan recursos y por qué).
    - Proponer 2 escenarios "what-if" concretos (cambios numéricos en parámetros) y predecir el efecto en el objetivo.
    - Sugerir mejoras al modelo (restricciones adicionales, variables útiles, cambios de objetivo) y priorizarlas.
    - Señalar riesgos, supuestos fuertes y posibles artefactos numéricos (p. ej. soluciones fraccionales si aplica).
    - Entregar al final un resumen compacto en JSON con claves: `veredicto`, `binding_constraints`, `recommended_changes` (lista con estimación de impacto), `scenarios`.

    Responde en español. Primero una explicación detallada, luego el JSON separado por una línea con tres guiones.
    """
    if client is None:
        return "Agente IA no configurado en este entorno."

    resultados_json = json.dumps(resultados, indent=2, ensure_ascii=False)

    prompt = f"""
Eres un experto senior en Investigación de Operaciones y Programación Entera Mixta (MIP).

Se acaba de resolver el siguiente problema con PuLP.

PROBLEMA: {nombre_problema}

RESULTADOS (JSON):
{resultados_json}

TAREAS (ordénalas y numéralas en la respuesta):
1) Verificación numérica: confirma que todas las restricciones del modelo se cumplen. Para cada restricción clave, muestra el valor LHS y RHS y di si se cumple o no.
2) Identificación de restricciones activas (binding): nombra las restricciones que limitan la solución y explica su efecto.
3) Explicación intuitiva: interpreta la solución en términos prácticos (qué decisiones tomó el modelo y por qué).
4) Sensibilidad y escenarios: propone 2 escenarios concretos (por ejemplo, +10% presupuesto, quitar costo fijo X, aumentar demanda Y) con valores numéricos, y predice cómo cambiaría el objetivo y las decisiones (dirección y estimación si posible).
5) Recomendaciones de mejora del modelo: sugiere hasta 4 cambios (nuevas restricciones, variables, objetivos multi-criterio), priorízalos y estima el impacto esperado.
6) Riesgos y controles: indica supuestos peligrosos y pruebas que el analista debe ejecutar para aumentar confianza.
7) Resumen JSON compacto (al final): debe contener las claves `veredicto` ("válida"/"problema"), `binding_constraints` (lista), `recommended_changes` (lista de objetos {{ "change": "descripción", "priority": "alta/baja", "est_impact": "valor" }}), `scenarios` (lista con {{ "nombre": "nombre", "change": "valor", "expected_effect": "efecto" }}).

Responde primero con la explicación detallada en español, y luego deja una línea con `---` y el bloque JSON final.
"""

    try:
        response = client.models.generate_content(model="gemini-3.5-flash", contents=prompt)
        return getattr(response, 'text', str(response))
    except Exception as e:
        return f"Error al consultar agente IA: {e}"

## Ejercicio 1 — Expansión de bodegas logísticas

# Ejercicio 1: Expansión de bodegas logísticas

## 1. Enunciado
Una empresa logística debe decidir en cuáles ciudades (Bogotá, Medellín, Cali) abrir nuevas instalaciones y qué cantidad de carga almacenar en cada una. Abrir una bodega tiene un costo fijo elevado, pero permite obtener ganancias por cada tonelada almacenada. Se cuenta con un presupuesto máximo de 100,000 y se debe cumplir con una demanda mínima de 180 toneladas. Solo se pueden abrir un máximo de 2 bodegas.

* **Bogotá:** Capacidad 200t, Costo Fijo 50,000, Ganancia/t 300
* **Medellín:** Capacidad 150t, Costo Fijo 40,000, Ganancia/t 350
* **Cali:** Capacidad 100t, Costo Fijo 35,000, Ganancia/t 280

## 2. Planteamiento Matemático
* **Variables:** $y_{bog}, y_{med}, y_{cal} \in \{0, 1\}$ (Apertura) y $x_{bog}, x_{med}, x_{cal} \ge 0$ (Toneladas).
* **Función Objetivo:** $$Max \ Z = 300x_{bog} + 350x_{med} + 280x_{cal} - 50000y_{bog} - 40000y_{med} - 35000y_{cal}$$
* **Restricciones:**
  1. $50000y_{bog} + 40000y_{med} + 35000y_{cal} \le 100000$ (Presupuesto)
  2. $y_{bog} + y_{med} + y_{cal} \le 2$ (Máximo de aperturas)
  3. $x_{bog} + x_{med} + x_{cal} \ge 180$ (Demanda mínima)
  4. $x_{bog} \le 200y_{bog}$, $x_{med} \le 150y_{med}$, $x_{cal} \le 100y_{cal}$ (Capacidad)

---

## 3. Resolución analítica paso a paso
Dado que tenemos 3 variables binarias, existen teóricamente $2^3 = 8$ escenarios posibles de apertura. Evaluaremos las ramas lógicas eliminando las que violan las restricciones.

### Paso 3.1: Descartar escenarios inviables
* **Escenario abrir 0 bodegas (0,0,0):** Se descarta de inmediato porque no se cumple la demanda mínima de 180 toneladas.
* **Escenarios de abrir 3 bodegas (1,1,1):** Se descarta por la restricción de aperturas ($y \le 2$) y por presupuesto (sumaría 125,000 > 100,000).

### Paso 3.2: Evaluar escenarios de abrir 1 bodega
Si solo abrimos una, esta debe ser capaz de soportar las 180 toneladas por sí sola.
* **Solo Cali (0,0,1):** Capacidad máxima 100t. No cumple la demanda (100 < 180). **Descartado.**
* **Solo Medellín (0,1,0):** Capacidad máxima 150t. No cumple la demanda (150 < 180). **Descartado.**
* **Solo Bogotá (1,0,0):** Capacidad máxima 200t. Sí cumple (200 $\ge$ 180). El costo fijo es 50,000 (cumple presupuesto).
  * *Optimización continua:* Para maximizar la ganancia, asignamos a Bogotá la cantidad máxima posible: $x_{bog} = 200$.
  * *Cálculo de Z:* $300(200) - 50000 = 60000 - 50000 = 10,000$.

### Paso 3.3: Evaluar escenarios de abrir 2 bodegas
* **Bogotá y Cali (1,0,1):**
  * *Presupuesto:* 50,000 + 35,000 = 85,000 $\le$ 100,000. (Válido).
  * *Optimización continua:* Bogotá paga más (300) que Cali (280). Llenamos Bogotá primero ($x_{bog} = 200$) y luego llenamos Cali ($x_{cal} = 100$).
  * *Cálculo de Z:* $300(200) + 280(100) - 85000 = 60000 + 28000 - 85000 = 3,000$.
* **Medellín y Cali (0,1,1):**
  * *Presupuesto:* 40,000 + 35,000 = 75,000 $\le$ 100,000. (Válido).
  * *Optimización continua:* Medellín paga más (350). Llenamos Medellín ($x_{med} = 150$) y luego Cali ($x_{cal} = 100$).
  * *Cálculo de Z:* $350(150) + 280(100) - 75000 = 52500 + 28000 - 75000 = 5,500$.
* **Bogotá y Medellín (1,1,0):**
  * *Presupuesto:* 50,000 + 40,000 = 90,000 $\le$ 100,000. (Válido).
  * *Optimización continua:* Llenamos ambas a su máxima capacidad porque su margen unitario es altamente positivo. $x_{bog} = 200$, $x_{med} = 150$. La carga total es 350t (cumple $\ge$ 180).
  * *Cálculo de Z:* $300(200) + 350(150) - 90000 = 60000 + 52500 - 90000 = 22,500$.

### Paso 3.4: Conclusión
Al comparar los resultados de todos los escenarios factibles, la configuración **Bogotá y Medellín** arroja la máxima ganancia neta.
* **Solución óptima:** $y_{bog}=1, y_{med}=1, y_{cal}=0$.
* **Asignación:** $x_{bog} = 200, x_{med} = 150, x_{cal} = 0$.
* **Z máximo:** $22,500.

In [55]:
def resolver_problema_1():
    modelo = pulp.LpProblem("Expansion_Bodegas", sense=pulp.LpMaximize)

    y_bog = pulp.LpVariable("y_bogota", cat='Binary')
    y_med = pulp.LpVariable("y_medellin", cat='Binary')
    y_cal = pulp.LpVariable("y_cali", cat='Binary')

    x_bog = pulp.LpVariable("x_bogota", lowBound=0, cat='Continuous')
    x_med = pulp.LpVariable("x_medellin", lowBound=0, cat='Continuous')
    x_cal = pulp.LpVariable("x_cali", lowBound=0, cat='Continuous')

    modelo += (
        300*x_bog + 350*x_med + 280*x_cal
        - 50000*y_bog - 40000*y_med - 35000*y_cal
    ), "Ganancia_Total"

    modelo += (50000*y_bog + 40000*y_med + 35000*y_cal <= 100000), "Presupuesto"
    modelo += (y_bog + y_med + y_cal <= 2), "Max_Bodegas"
    modelo += (x_bog + x_med + x_cal >= 180), "Demanda_Minima"
    modelo += (x_bog <= 200 * y_bog), "Cap_Bogota"
    modelo += (x_med <= 150 * y_med), "Cap_Medellin"
    modelo += (x_cal <= 100 * y_cal), "Cap_Cali"

    solver = pulp.PULP_CBC_CMD(msg=0)
    modelo.solve(solver)

    estado = pulp.LpStatus[modelo.status]

    resultados = {
        'estado_solucion': estado,
        'es_optima': estado == 'Optimal',
        'ganancia_optima_total': round(pulp.value(modelo.objective), 2),
        'decisiones_apertura': {
            'Bogota': {
                'abrir': bool(round(pulp.value(y_bog))),
                'valor_binario': int(round(pulp.value(y_bog))),
                'toneladas_almacenar': round(pulp.value(x_bog), 2)
            },
            'Medellin': {
                'abrir': bool(round(pulp.value(y_med))),
                'valor_binario': int(round(pulp.value(y_med))),
                'toneladas_almacenar': round(pulp.value(x_med), 2)
            },
            'Cali': {
                'abrir': bool(round(pulp.value(y_cal))),
                'valor_binario': int(round(pulp.value(y_cal))),
                'toneladas_almacenar': round(pulp.value(x_cal), 2)
            }
        },
    }

    # Validación matemática
    ganancia_calculada = (
        300 * pulp.value(x_bog) + 350 * pulp.value(x_med) + 280 * pulp.value(x_cal)
        - 50000 * pulp.value(y_bog) - 40000 * pulp.value(y_med) - 35000 * pulp.value(y_cal)
    )

    resultados['validacion_matematica'] = {
        'ganancia_reportada_por_solver': round(pulp.value(modelo.objective), 2),
        'ganancia_calculada_manualmente': round(ganancia_calculada, 2),
        'validacion_exitosa': abs(pulp.value(modelo.objective) - ganancia_calculada) < 0.01
    }

    return resultados

# Ejercicio 2: Selección de proyectos de inversión

## 1. Enunciado
Seleccionar proyectos (Alpha, Beta, Gamma, Delta) y determinar la inversión en cada uno bajo restricciones de presupuesto global (200,000), exclusión mutua y límites mínimos/máximos de capital por proyecto. El objetivo es maximizar el ROI neto.

* **Alpha:** Fijo 10k, Inv (20k a 80k), ROI 15%
* **Beta:** Fijo 15k, Inv (30k a 100k), ROI 18%
* **Gamma:** Fijo 8k, Inv (15k a 60k), ROI 12%
* **Delta:** Fijo 20k, Inv (50k a 120k), ROI 20%. *Excluyente con Alpha.*

## 2. Planteamiento Matemático
* **Variables:** $z_a, z_b, z_g, z_d \in \{0, 1\}$ y $inv_a, inv_b, inv_g, inv_d \ge 0$.
* **Función Objetivo:** $$Max \ Z = 0.15inv_a + 0.18inv_b + 0.12inv_g + 0.20inv_d - 10000z_a - 15000z_b - 8000z_g - 20000z_d$$
* **Restricciones principales:**
  1. $\sum Costos Fijos + \sum Inversiones \le 200000$
  2. $z_a + z_d \le 1$ (Exclusión Delta-Alpha)

---

## 3. Resolución analítica paso a paso ("A mano")

El objetivo se comporta como un problema de la mochila (Knapsack Problem). Tenemos una cantidad de capital finito y debemos asignarlo a las opciones que generen mayor tasa de retorno, descontando el costo de entrada (Costo Fijo).

Analicemos la rentabilidad pura de los proyectos:
1. **Delta:** 20% (Requiere alto costo de entrada, excluye a Alpha).
2. **Beta:** 18% (Excelente balance, límite alto de inversión de 100k).
3. **Alpha:** 15%.
4. **Gamma:** 12% (Poco atractivo comparativamente).

### Paso 3.1: Escenario priorizando a Delta (La tasa más alta)
Dado que Delta tiene un ROI de 20%, intentamos un portafolio alrededor de él.
* Si elegimos **Delta**, por exclusión ($z_a + z_d \le 1$), **Alpha queda fuera**.
* Intentamos combinarlo con **Beta** (18% ROI) para maximizar el uso del presupuesto.
* *Escenario (Delta + Beta):* $z_d = 1, z_b = 1, z_a = 0, z_g = 0$.
* Presupuesto consumido por costos fijos: $20,000 + 15,000 = 35,000$.
* Presupuesto disponible para invertir: $200,000 - 35,000 = 165,000$.
* Asignación: Llenamos primero el de mayor tasa (Delta). Le asignamos su tope máximo: $inv_d = 120,000$.
* Quedan $45,000$. Se los asignamos a Beta ($inv_b = 45,000$, cumple el mínimo de 30k).
* *Cálculo de Z:* $0.20(120k) + 0.18(45k) - 35k = 24,000 + 8,100 - 35,000 = -2,900$.
* *Análisis:* Este portafolio da pérdida. Los altos costos fijos de Delta (20k) no se logran compensar a pesar del alto porcentaje, porque castigan demasiado el capital disponible para invertir.

### Paso 3.2: Escenario priorizando Alpha y Beta
Si descartamos a Delta, podemos usar a Alpha (15%) y Beta (18%).
* *Escenario (Alpha + Beta):* $z_a = 1, z_b = 1, z_d = 0, z_g = 0$.
* Presupuesto consumido por costos fijos: $10,000 + 15,000 = 25,000$.
* Presupuesto disponible para invertir: $200,000 - 25,000 = 175,000$.
* Asignación: Llenamos primero el de mayor tasa (Beta). Le asignamos su tope máximo: $inv_b = 100,000$.
* Quedan $75,000$. Se los asignamos a Alpha ($inv_a = 75,000$, cumple rango 20k-80k).
* *Cálculo de Z:* $0.15(75k) + 0.18(100k) - 25k = 11,250 + 18,000 - 25,000 = 29,250 - 25,000 = 4,250$.

### Paso 3.3: Evaluar inclusión de Gamma
¿Qué pasa si al escenario anterior le agregamos Gamma (12%)?
* Costos fijos suben a $33,000$. Quedan $167,000$ para invertir.
* Llenamos Beta (100k), nos obligan a darle al menos 15k a Gamma. Quedan 52k para Alpha.
* *Cálculo de Z:* $0.18(100k) + 0.15(52k) + 0.12(15k) - 33k = 18,000 + 7,800 + 1,800 - 33,000 = -5,400$.
* *Análisis:* Añadir Gamma destruye valor porque su retorno es muy bajo y quita capital a Alpha, además de agregar un costo fijo de 8,000.

### Paso 3.4: Conclusión
El balance óptimo entre costos hundidos de entrada y tasas de retorno lo proporcionan los proyectos Alpha y Beta en conjunto.
* **Solución óptima:** $z_a=1, z_b=1, z_g=0, z_d=0$.
* **Inversión:** $inv_a = 75,000, inv_b = 100,000$.
* **Z máximo:** $4,250$.

In [56]:
def resolver_problema_2():
    modelo = pulp.LpProblem("Seleccion_Proyectos", sense=pulp.LpMaximize)

    z_a = pulp.LpVariable("z_alpha", cat='Binary')
    z_b = pulp.LpVariable("z_beta", cat='Binary')
    z_g = pulp.LpVariable("z_gamma", cat='Binary')
    z_d = pulp.LpVariable("z_delta", cat='Binary')

    inv_a = pulp.LpVariable("inv_alpha", lowBound=0, cat='Continuous')
    inv_b = pulp.LpVariable("inv_beta", lowBound=0, cat='Continuous')
    inv_g = pulp.LpVariable("inv_gamma", lowBound=0, cat='Continuous')
    inv_d = pulp.LpVariable("inv_delta", lowBound=0, cat='Continuous')

    modelo += (
        0.15*inv_a + 0.18*inv_b + 0.12*inv_g + 0.20*inv_d
        - 10000*z_a - 15000*z_b - 8000*z_g - 20000*z_d
    ), "ROI_Total"

    modelo += (
        10000*z_a + 15000*z_b + 8000*z_g + 20000*z_d
        + inv_a + inv_b + inv_g + inv_d <= 200000
    ), "Presupuesto_Total"

    modelo += (z_a + z_b + z_g + z_d <= 3), "Max_Proyectos"
    modelo += (z_d + z_a <= 1), "Exclusion_Delta_Alpha"

    modelo += (inv_a >= 20000 * z_a), "Inv_Min_Alpha"
    modelo += (inv_b >= 30000 * z_b), "Inv_Min_Beta"
    modelo += (inv_g >= 15000 * z_g), "Inv_Min_Gamma"
    modelo += (inv_d >= 50000 * z_d), "Inv_Min_Delta"

    modelo += (inv_a <= 80000  * z_a), "Inv_Max_Alpha"
    modelo += (inv_b <= 100000 * z_b), "Inv_Max_Beta"
    modelo += (inv_g <= 60000  * z_g), "Inv_Max_Gamma"
    modelo += (inv_d <= 120000 * z_d), "Inv_Max_Delta"

    modelo += (inv_a + inv_b + inv_g + inv_d >= 60000), "Inversion_Minima"

    solver = pulp.PULP_CBC_CMD(msg=0)
    modelo.solve(solver)

    estado = pulp.LpStatus[modelo.status]

    resultados = {
        'estado_solucion': estado,
        'es_optima': estado == 'Optimal',
        'roi_neto_optimo': round(pulp.value(modelo.objective), 2),
        'proyectos': {
            'Alpha': {
                'activar': bool(round(pulp.value(z_a))),
                'inversion': round(pulp.value(inv_a), 2),
                'roi_generado': round(0.15 * pulp.value(inv_a), 2)
            },
            'Beta': {
                'activar': bool(round(pulp.value(z_b))),
                'inversion': round(pulp.value(inv_b), 2),
                'roi_generado': round(0.18 * pulp.value(inv_b), 2)
            },
            'Gamma': {
                'activar': bool(round(pulp.value(z_g))),
                'inversion': round(pulp.value(inv_g), 2),
                'roi_generado': round(0.12 * pulp.value(inv_g), 2)
            },
            'Delta': {
                'activar': bool(round(pulp.value(z_d))),
                'inversion': round(pulp.value(inv_d), 2),
                'roi_generado': round(0.20 * pulp.value(inv_d), 2)
            }
        }
    }

    roi_calculado = (
        0.15*pulp.value(inv_a) + 0.18*pulp.value(inv_b) +
        0.12*pulp.value(inv_g) + 0.20*pulp.value(inv_d) -
        10000*pulp.value(z_a) - 15000*pulp.value(z_b) - 8000*pulp.value(z_g) - 20000*pulp.value(z_d)
    )

    resultados['validacion_matematica'] = {
        'roi_reportado_por_solver': round(pulp.value(modelo.objective), 2),
        'roi_calculado_manualmente': round(roi_calculado, 2),
        'validacion_exitosa': abs(pulp.value(modelo.objective) - roi_calculado) < 0.01
    }

    return resultados

# Ejercicio 3: Programación de producción en planta manufacturera

## 1. Enunciado
Decidir qué líneas de producción (L1, L2) encender y cuántos productos (A, B, C) asignar a cada una. Se busca maximizar la ganancia neta respetando la capacidad de consumo de materia prima por línea y los topes de demanda.
* **Precios:** A(120), B(150), C(90).
* **Demandas máximas:** A(80), B(60), C(100).
* **Línea 1 (CF: 5000, Cap: 300):** Consumo unitario -> A(2), B(3), C(1).
* **Línea 2 (CF: 6000, Cap: 250):** Consumo unitario -> A(1.5), B(2), C(2.5).

## 2. Planteamiento Matemático
* **Variables:** $w_1, w_2 \in \{0, 1\}$ (Encendido) y $p_{ij} \ge 0$ (Producción).
* **Función Objetivo:** $$Max \ Z = 120(p_{a1}+p_{a2}) + 150(p_{b1}+p_{b2}) + 90(p_{c1}+p_{c2}) - 5000w_1 - 6000w_2$$
* **Restricciones principales:**
  1. $2p_{a1} + 3p_{b1} + 1p_{c1} \le 300w_1$
  2. $1.5p_{a2} + 2p_{b2} + 2.5p_{c2} \le 250w_2$
  3. Límite de ventas por producto.

---

## 3. Resolución analítica paso a paso

Para resolver este problema manualmente, calculamos la **eficiencia del recurso** de cada producto en cada línea (Margen aportado / Materia prima consumida). Esto nos dirá a qué línea debemos asignar qué producto.

* **Eficiencias en Línea 1 (Precio / Consumo L1):**
  * Producto A: $120 / 2 = 60$
  * Producto B: $150 / 3 = 50$
  * Producto C: $90 / 1 = 90$ **(¡La más alta!)**
* **Eficiencias en Línea 2 (Precio / Consumo L2):**
  * Producto A: $120 / 1.5 = 80$
  * Producto B: $150 / 2 = 75$
  * Producto C: $90 / 2.5 = 36$

### Paso 3.1: Estrategia de Asignación Óptima (Ambas líneas encendidas)
Supongamos que activamos ambas líneas ($w_1 = 1, w_2 = 1$). Queremos fabricar el máximo de todos los productos (A=80, B=60, C=100) distribuyéndolos donde sean más eficientes.
* **Asignando C:** Su eficiencia es masiva en la Línea 1 (90) y terrible en la 2 (36). Enviamos toda la demanda de C a la L1.
  * $p_{c1} = 100$. Consumo en L1: $100 \times 1 = 100$. Quedan 200 de capacidad en L1.
* **Asignando B:** Su eficiencia es mejor en la Línea 2 (75 vs 50). Enviamos toda la demanda de B a la L2.
  * $p_{b2} = 60$. Consumo en L2: $60 \times 2 = 120$. Quedan 130 de capacidad en L2.
* **Asignando A:** Su eficiencia es mejor en la Línea 2 (80 vs 60), pero tenemos espacio de sobra en ambas. 
  * Si lo enviamos a L1: $p_{a1} = 80$. Consumo L1: $80 \times 2 = 160$. (Quedaría $100 + 160 = 260 \le 300$). Factible.
  * Si lo enviamos a L2: $p_{a2} = 80$. Consumo L2: $80 \times 1.5 = 120$. (Quedaría $120 + 120 = 240 \le 250$). Factible.
* El modelo MIP del solver eligió hacer A y C en la Línea 1, y B en la Línea 2, lo cual es perfectamente válido y genera el mismo ingreso máximo, ya que se cubren exactamente las demandas máximas (80, 60, 100).

### Paso 3.2: Cálculo de la rentabilidad máxima (Ambas abiertas)
* Ingresos por ventas = $120(80) + 150(60) + 90(100) = 9600 + 9000 + 9000 = 27,600$.
* Costos fijos (L1 + L2) = $5000 + 6000 = 11,000$.
* **Z = $27,600 - 11,000 = 16,600$.**

### Paso 3.3: Comprobación de escenario apagando una línea
¿Podríamos ganar más si apagamos una línea para ahorrarnos su costo fijo?
* **Si apagamos L2 (Ahorro de 6000):** Solo disponemos de 300 unidades de capacidad en L1. Priorizamos C (100 de capacidad usada), luego A (160 usada). Llevamos 260. Quedan 40. Para B alcanzamos a producir $40/3 = 13.33$ unidades.
  * Ingreso: $120(80) + 150(13.33) + 90(100) = 9600 + 2000 + 9000 = 20,600$.
  * Costo: 5000.
  * *Z = 15,600.* (Inferior a 16,600).
* **Si apagamos L1 (Ahorro de 5000):** Solo disponemos de 250 unidades en L2. L2 es terrible para C. Priorizamos A (120 de capacidad), luego B (120 usada). Llevamos 240. Quedan 10 de capacidad, que alcanzan para 4 unidades de C ($10/2.5$).
  * Ingreso: $120(80) + 150(60) + 90(4) = 9600 + 9000 + 360 = 18,960$.
  * Costo: 6000.
  * *Z = 12,960.* (Muy inferior).

### Paso 3.4: Conclusión
Es necesario y altamente rentable pagar el costo de encendido de ambas maquinarias.
* **Solución óptima:** $w_1 = 1, w_2 = 1$.
* **Asignación:** Línea 1 fabrica A(80) y C(100). Línea 2 fabrica B(60).
* **Z máximo:** $16,600$.

In [57]:
def resolver_problema_3():
    modelo = pulp.LpProblem("Produccion_Manufacturera", sense=pulp.LpMaximize)

    w1 = pulp.LpVariable("w_linea1", cat='Binary')
    w2 = pulp.LpVariable("w_linea2", cat='Binary')

    p_a1 = pulp.LpVariable("p_A_L1", lowBound=0, cat='Continuous')
    p_b1 = pulp.LpVariable("p_B_L1", lowBound=0, cat='Continuous')
    p_c1 = pulp.LpVariable("p_C_L1", lowBound=0, cat='Continuous')
    p_a2 = pulp.LpVariable("p_A_L2", lowBound=0, cat='Continuous')
    p_b2 = pulp.LpVariable("p_B_L2", lowBound=0, cat='Continuous')
    p_c2 = pulp.LpVariable("p_C_L2", lowBound=0, cat='Continuous')

    precio_a, precio_b, precio_c = 120, 150, 90

    modelo += (
        precio_a * (p_a1 + p_a2) +
        precio_b * (p_b1 + p_b2) +
        precio_c * (p_c1 + p_c2) -
        5000*w1 - 6000*w2
    ), "Ganancia_Produccion"

    modelo += (w1 + w2 >= 1), "Minimo_Una_Linea"
    modelo += (2*p_a1 + 3*p_b1 + 1*p_c1 <= 300 * w1), "Materia_Prima_L1"
    modelo += (1.5*p_a2 + 2*p_b2 + 2.5*p_c2 <= 250 * w2), "Materia_Prima_L2"
    modelo += (p_a1 + p_a2 <= 80), "Demanda_Max_A"
    modelo += (p_b1 + p_b2 <= 60), "Demanda_Max_B"
    modelo += (p_c1 + p_c2 <= 100), "Demanda_Max_C"
    modelo += (p_a1 + p_a2 + p_b1 + p_b2 + p_c1 + p_c2 >= 100), "Produccion_Minima"

    solver = pulp.PULP_CBC_CMD(msg=0)
    modelo.solve(solver)

    estado = pulp.LpStatus[modelo.status]

    total_a = pulp.value(p_a1) + pulp.value(p_a2)
    total_b = pulp.value(p_b1) + pulp.value(p_b2)
    total_c = pulp.value(p_c1) + pulp.value(p_c2)

    resultados = {
        'estado_solucion': estado,
        'es_optima': estado == 'Optimal',
        'ganancia_neta_optima': round(pulp.value(modelo.objective), 2),
        'lineas_activas': {
            'Linea_1': {
                'activa': bool(round(pulp.value(w1))),
                'produccion_A': round(pulp.value(p_a1), 2),
                'produccion_B': round(pulp.value(p_b1), 2),
                'produccion_C': round(pulp.value(p_c1), 2),
            },
            'Linea_2': {
                'activa': bool(round(pulp.value(w2))),
                'produccion_A': round(pulp.value(p_a2), 2),
                'produccion_B': round(pulp.value(p_b2), 2),
                'produccion_C': round(pulp.value(p_c2), 2),
            }
        },
        'totales_por_producto': {
            'Producto_A': {'unidades_totales': round(total_a, 2), 'ingreso': round(precio_a*total_a, 2)},
            'Producto_B': {'unidades_totales': round(total_b, 2), 'ingreso': round(precio_b*total_b, 2)},
            'Producto_C': {'unidades_totales': round(total_c, 2), 'ingreso': round(precio_c*total_c, 2)}
        }
    }

    ganancia_calc = (
        precio_a*(pulp.value(p_a1)+pulp.value(p_a2)) +
        precio_b*(pulp.value(p_b1)+pulp.value(p_b2)) +
        precio_c*(pulp.value(p_c1)+pulp.value(p_c2)) -
        5000*pulp.value(w1) - 6000*pulp.value(w2)
    )

    resultados['validacion_matematica'] = {
        'ganancia_reportada_por_solver': round(pulp.value(modelo.objective), 2),
        'ganancia_calculada_manualmente': round(ganancia_calc, 2),
        'validacion_exitosa': abs(pulp.value(modelo.objective) - ganancia_calc) < 0.01
    }

    return resultados

## Ejecutar los tres ejercicios y ver resumen

La siguiente celda ejecuta las tres funciones y muestra un resumen sencillo. Si configuras `API_KEY`, también intentará consultar al agente IA (si la librería está disponible).

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True) 

# Obtienes la llave directamente del sistema
API_KEY = os.getenv("API_KEY")

def main():
    usar_ia = False
    client = None
    
    # Verificamos que la llave se haya cargado correctamente
    if not API_KEY:
        print("Error: No se encontró la API_KEY en el archivo .env")
        return
        
    client = crear_cliente_gemini(API_KEY)
    if client is not None:
        usar_ia = True

    res1 = resolver_problema_1()
    res2 = resolver_problema_2()
    res3 = resolver_problema_3()

    for i, (res, nombre) in enumerate([
        (res1, 'Expansión de Bodegas'),
        (res2, 'Selección de Proyectos'),
        (res3, 'Producción Manufacturera')
    ], 1):
        estado = 'VÁLIDA' if res['validacion_matematica']['validacion_exitosa'] else 'ERROR'
        print(f'Problema {i} ({nombre}): {estado} — Óptimo encontrado: {res["es_optima"]}')

    if usar_ia:
        for resultados, nombre in [(res1, 'Expansión de Bodegas'), (res2, 'Selección de Proyectos'), (res3, 'Producción Manufacturera')]:
            analisis = consultar_agente_ia(client, resultados, nombre)
            print(f"\n--- Análisis IA: {nombre} ---")
            print(analisis)


if __name__ == '__main__':
    main()

Problema 1 (Expansión de Bodegas): VÁLIDA — Óptimo encontrado: True
Problema 2 (Selección de Proyectos): VÁLIDA — Óptimo encontrado: True
Problema 3 (Producción Manufacturera): VÁLIDA — Óptimo encontrado: True

--- Análisis IA: Expansión de Bodegas ---
Hola. Como especialista senior en Investigación de Operaciones, he analizado en detalle los resultados obtenidos para el problema de **Expansión de Bodegas**. A continuación, presento la auditoría técnica, la interpretación del negocio, los escenarios de sensibilidad y las recomendaciones de nivel de producción.

---

### 1) Verificación Numérica de Restricciones
Para validar la viabilidad del modelo matemático resuelto por PuLP, contrastamos el valor del lado izquierdo (LHS) con el del lado derecho (RHS) de las restricciones clave bajo la solución óptima:

1. **Restricción de Activación de Capacidad ($x_i \le Capacity_i \cdot y_i$):**
   * **Bogotá:** LHS: $200.0$ t | RHS: $200.0 \cdot 1 = 200.0$ t. **Estado: Se cumple (Activa).**
   * 